# 02. Data Cleaning & ETL Pipeline
**Project:** Online Retail Sales Analytics — Capstone DVA  
**Dataset:** UCI Online Retail (541,909 rows, 8 columns)  
**Objective:** Transform the raw transactional dataset into a clean, analysis-ready dataset with engineered KPI columns for EDA and Tableau dashboarding.

---
### Pipeline Overview
| Step | Action | Type |
|------|--------|------|
| 1 | Load raw Excel dataset | Extract |
| 2 | Remove exact duplicate rows | Transform |
| 3 | Impute missing CustomerID → 'Guest' | Transform |
| 4 | Impute missing Description → 'Unknown' | Transform |
| 5 | Remove junk/system-generated rows | Transform |
| 6 | Drop corrupt rows (bad qty/price) | Transform |
| 7 | Flag cancellations with IsCancelled column | Transform |
| 8 | Date & Time feature engineering | Transform |
| 9 | Geographic segmentation (UK vs International) | Transform |
| 10 | Basket/Order level KPIs | Transform |
| 11 | Customer-level KPIs (Repeat, AvgOrderValue) | Transform |
| 12 | RFM Customer Segmentation | Transform |
| 13 | Export clean dataset to data/processed/ | Load |

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

---
## STEP 1 — Load Raw Dataset

In [ ]:
raw_path = '../data/raw/Online Retail.xlsx'
print(f'Loading: {raw_path}')
df = pd.read_excel(raw_path)

print(f'\nShape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Raw dataset health check
print('=== RAW DATASET HEALTH CHECK ===')
print(f'Total Rows: {len(df):,}')
print(f'\nNull Values per Column:')
print(df.isnull().sum())
print(f'\nData Types:')
print(df.dtypes)
print(f'\nDuplicate Rows: {df.duplicated().sum():,}')

---
## STEP 2 — Remove Exact Duplicate Rows
**Reason:** Database glitches in the original system caused identical transactions to be logged multiple times. These inflate product quantities and revenue figures.

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Duplicates removed: {before - after:,}')
print(f'Rows remaining: {after:,}')

---
## STEP 3 — Handle Missing CustomerID
**Problem:** 135,080 rows (24.9%) have no CustomerID — representing guest checkouts.  
**Decision:** Instead of dropping these rows (which represent £1.75M+ in real revenue), we impute them with the label `'Guest'`. This preserves all valid transaction data. Guest rows will be excluded only from customer-specific analyses like RFM.

In [ ]:
missing_cid = df['CustomerID'].isna().sum()
print(f'Missing CustomerIDs before imputation: {missing_cid:,}')

df['CustomerID'] = df['CustomerID'].fillna(0).astype(int).astype(str)
df['CustomerID'] = df['CustomerID'].replace('0', 'Guest')

print(f'Guest rows after imputation: {(df["CustomerID"] == "Guest").sum():,}')
print(f'Missing CustomerIDs after imputation: {df["CustomerID"].isna().sum()}')

---
## STEP 4 — Handle Missing Description
**Problem:** 1,454 rows have no product description.  
**Decision:** Fill with `'Unknown'` to retain the transaction data.

In [ ]:
print(f'Missing Descriptions before: {df["Description"].isna().sum():,}')
df['Description'] = df['Description'].fillna('Unknown').astype(str).str.strip().str.upper()
print(f'Missing Descriptions after: {df["Description"].isna().sum()}')

---
## STEP 5 — Remove Junk System Rows
**Problem:** Some rows contain internal accounting/system entries like `POSTAGE`, `AMAZON FEE`, or `Manual` instead of actual products. These corrupt any product-level revenue analysis.

In [ ]:
junk_labels = ['POSTAGE', 'DOTCOM POSTAGE', 'CRUK COMMISSION', 'MANUAL', 'BANK CHARGES', 'AMAZON FEE']
before = len(df)
df = df[~df['Description'].isin(junk_labels)]
print(f'Junk rows removed: {before - len(df):,}')
print(f'Rows remaining: {len(df):,}')

---
## STEP 6 — Drop Corrupt Rows (Bad Quantity / Zero Price)
**Problem:** Some rows have negative quantities that are NOT cancellations (data entry errors), and some have a £0.00 or negative UnitPrice (bad debt adjustments not suitable for revenue analysis).

In [ ]:
before = len(df)
is_cancel = df['InvoiceNo'].astype(str).str.startswith('C')

# Remove negative quantities that are NOT cancellations (data errors)
df = df[~((df['Quantity'] < 0) & (~is_cancel))]

# Remove zero or negative prices
df = df[df['UnitPrice'] > 0]

print(f'Corrupt rows removed: {before - len(df):,}')
print(f'Rows remaining: {len(df):,}')

---
## STEP 7 — Flag Cancellations (Do NOT Drop)
**Decision:** We keep cancellation rows in the dataset but flag them with `IsCancelled = True`. This allows us to calculate a **Cancellation Rate KPI** in Tableau, which is a key business metric.

In [ ]:
df['IsCancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')
print(f'Active orders   : {(~df["IsCancelled"]).sum():,}')
print(f'Cancelled orders: {df["IsCancelled"].sum():,}')
print(f'Cancellation Rate: {df["IsCancelled"].mean()*100:.2f}%')

---
## STEP 8 — Date & Time Feature Engineering
These columns are essential for time-series trend analysis in Tableau.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Revenue per line item
df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']

# Time breakdown columns
df['InvoiceYearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')   # e.g., 2011-10
df['MonthName']        = df['InvoiceDate'].dt.strftime('%B')       # e.g., October
df['Quarter']          = 'Q' + df['InvoiceDate'].dt.quarter.astype(str)  # e.g., Q4
df['MonthNumber']      = df['InvoiceDate'].dt.month                  # e.g., 10 (for sorting months correctly in Tableau)
df['Year']             = df['InvoiceDate'].dt.year                   # e.g., 2011 (for year-over-year pivot comparisons)
df['DayOfWeek']        = df['InvoiceDate'].dt.day_name()           # e.g., Monday
df['Hour']             = df['InvoiceDate'].dt.hour                 # e.g., 14

def time_of_day(h):
    if 5 <= h < 12:    return 'Morning'
    elif 12 <= h < 17: return 'Afternoon'
    elif 17 <= h < 21: return 'Evening'
    else:              return 'Night'

df['TimeOfDay'] = df['Hour'].apply(time_of_day)

print('Date & time columns created successfully.')
df[['InvoiceDate','TotalRevenue','InvoiceYearMonth','MonthName','Quarter','DayOfWeek','Hour','TimeOfDay']].head()

---
## STEP 9 — Geographic Segmentation
UK vs International split — essential for understanding market distribution in Tableau.

In [ ]:
df['IsUK'] = df['Country'].apply(lambda x: 'UK' if x == 'United Kingdom' else 'International')

print('Geographic distribution:')
print(df['IsUK'].value_counts())

---
## STEP 10 — Basket / Order Level KPIs
Segments each invoice by total spend into Small, Medium, or Large baskets. Useful for customer spending behavior analysis in Tableau.

In [ ]:
basket = df.groupby('InvoiceNo').agg(
    BasketRevenue = ('TotalRevenue', 'sum'),
    ItemsPerOrder = ('Quantity', 'sum')
).reset_index()

def basket_size(v):
    if v < 50:     return 'Small (< £50)'
    elif v <= 200: return 'Medium (£50-£200)'
    else:          return 'Large (> £200)'

basket['BasketSize'] = basket['BasketRevenue'].apply(basket_size)
df = df.merge(basket[['InvoiceNo', 'BasketSize', 'ItemsPerOrder']], on='InvoiceNo', how='left')

print('Basket Size distribution:')
print(df['BasketSize'].value_counts())

---
## STEP 11 — Customer Level KPIs (Repeat Buyer, Avg Order Value)
Identifies whether each customer is a one-time or repeat buyer — a key retention KPI for the Tableau dashboard.

In [ ]:
customer_stats = df[df['CustomerID'] != 'Guest'].groupby('CustomerID').agg(
    TotalOrders = ('InvoiceNo', 'nunique'),
    TotalSpend  = ('TotalRevenue', 'sum')
).reset_index()

customer_stats['AvgOrderValue']    = (customer_stats['TotalSpend'] / customer_stats['TotalOrders']).round(2)
customer_stats['IsRepeatCustomer'] = customer_stats['TotalOrders'].apply(lambda x: 'Repeat' if x > 1 else 'One-Time')

df = df.merge(customer_stats[['CustomerID', 'AvgOrderValue', 'IsRepeatCustomer']], on='CustomerID', how='left')
df['IsRepeatCustomer'] = df['IsRepeatCustomer'].fillna('Guest')
df['AvgOrderValue']    = df['AvgOrderValue'].fillna(0)

print('Customer Type Distribution:')
print(df['IsRepeatCustomer'].value_counts())

---
## STEP 12 — RFM Customer Segmentation
**RFM = Recency + Frequency + Monetary Value**  
This is an industry-standard model for segmenting customers into groups like Champions, Loyal Customers, At-Risk, and Lost. It scores each customer on a 1–4 scale in each dimension.

This is a **advanced capstone-level analysis** and will power the Customer Segmentation view in the Tableau dashboard.

In [ ]:
# Only real customers on non-cancelled orders
rfm_df = df[(df['CustomerID'] != 'Guest') & (df['IsCancelled'] == False)].copy()
snapshot_date = rfm_df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = rfm_df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('TotalRevenue', 'sum')
).reset_index()

# Score each dimension 1–4
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

def rfm_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:   return 'Champion'
    elif r >= 3 and f >= 3:             return 'Loyal Customer'
    elif r >= 3 and f <= 2:             return 'Potential Loyalist'
    elif r == 2 and f >= 2:             return 'At Risk'
    elif r <= 1 and f >= 3:             return 'Cant Lose Them'
    elif r <= 1 and f <= 1:             return 'Lost'
    else:                               return 'Needs Attention'

rfm['CustomerSegment'] = rfm.apply(rfm_segment, axis=1)

df = df.merge(rfm[['CustomerID', 'RFM_Score', 'CustomerSegment', 'Recency', 'Frequency', 'Monetary']],
              on='CustomerID', how='left')
df['CustomerSegment'] = df['CustomerSegment'].fillna('Guest')
df['RFM_Score']       = df['RFM_Score'].fillna('N/A')

print('Customer Segment Distribution:')
print(df['CustomerSegment'].value_counts())

---
## STEP 13 — Export Clean Dataset

In [ ]:
out_path = '../data/processed/cleaned_retail_data.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df.to_csv(out_path, index=False)

print('='*60)
print('  ETL PIPELINE COMPLETE!')
print(f'  Original rows   : 541,909')
print(f'  Final rows      : {len(df):,}')
print(f'  Original columns: 8')
print(f'  Final columns   : {len(df.columns)}')
print(f'  New columns     : {len(df.columns)-8}')
print(f'  Saved to        : {out_path}')
print('='*60)
df.head()